## Terminology Update

Some historical output logs in this notebook were captured before the current naming migration.
Current runtime terminology in the codebase is:
- `single_csv`
- `multi_csv`
- `ExecutionContext`


In [1]:
# Setup: add project root and pin Qwen 3.5 before src imports
import sys
import os

BASE = os.path.abspath(os.path.join('..'))
sys.path.insert(0, BASE)

from dotenv import load_dotenv

load_dotenv(os.path.join(BASE, '.env'))
os.environ['LLM_PROVIDER'] = 'surf'
os.environ['LLM_MODEL'] = 'Sehyo/Qwen3.5-122B-A10B-NVFP4'

from src.config import LLM_PROVIDER, get_model_name
from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context

import logging

# Setup logging for Jupyter notebook
# Force reconfigure by removing existing handlers
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add a fresh StreamHandler that outputs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

# STANDARD_NAME = "croissant_pangaea_standard"
STANDARD_NAME = "croissant_inaturalist_standard"
# STANDARD_NAME = "spatial_ecological"

print(f"LLM provider: {LLM_PROVIDER}")
print(f"LLM model: {get_model_name()}")
print("Imports OK")

LLM provider: surf
LLM model: Sehyo/Qwen3.5-122B-A10B-NVFP4
Imports OK


In [2]:
from src.orchestrator import run_metadata_extraction
inat_source = {
    'file_1': os.path.join(BASE, 'data/experiment/pangaea_datasets/897882.csv'),
}

# multi_source = {
#     'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
#     'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
#     'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/protoDT/annual_budburst_per_tree.csv'),
#     'occurrence': os.path.join(BASE, 'data/protoDT/budburst_climwin_input.csv'),
#     'temp': os.path.join(BASE, 'data/protoDT/temp_climwin_input.csv'),
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/cropharvest/cropharvest_cleaned_global.csv')
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/S2BMS/bms_presence_y-2018-2019_th-200.csv'),
# }

In [3]:
data_context = create_context(
    source=inat_source,
    name='test'
)
# metadata_standard=METADATA_STANDARDS['spatial_ecological']
metadata_standard = METADATA_STANDARDS[STANDARD_NAME]
orchestrator = Orchestrator(topology_name='single')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard,
    metadata_standard_name=STANDARD_NAME,
)

/tmp/ipykernel_57057/4155723445.py:7: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name='single')


2026-07-15 13:46:30,055 - INFO - root - PlanExecutor initialized with topology: single
2026-07-15 13:46:30,056 - INFO - root -   Players per step: 1
2026-07-15 13:46:30,056 - INFO - root -   Debate rounds: 0
2026-07-15 13:46:30,057 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-07-15 13:46:30,057 - INFO - root - Orchestrator initialized with topology: single
2026-07-15 13:46:30,058 - INFO - root - [ui] planning
2026-07-15 13:46:30,058 - INFO - root - ============================================================
2026-07-15 13:46:30,058 - INFO - root - GENERATING PLAN
2026-07-15 13:46:30,058 - INFO - root - Context: test
2026-07-15 13:46:30,058 - INFO - root - Context type: single_csv
2026-07-15 13:46:30,058 - INFO - root - Classified planning type: single_csv
2026-07-15 13:46:30,059 - INFO - root - Resources: ['file_1']
2026-07-15 13:46:30,059 - INFO - 

/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-15 13:46:58,223 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:46:58,241 - WARNING - root - Plan validation warning: Step 4 ('generate_metadata') has unmet dependencies. Missing artifacts: {'metadata_standard'}
2026-07-15 13:46:58,242 - INFO - root - Plan generated successfully!
2026-07-15 13:46:58,242 - INFO - root - Number of steps: 5
2026-07-15 13:46:58,242 - INFO - root -   Step 1: detect_spatial_columns (player: spatial_temporal_specialist)
2026-07-15 13:46:58,243 - INFO - root -   Step 2: detect_temporal_columns (player: spatial_temporal_specialist)
2026-07-15 13:46:58,243 - INFO - root -   Step 3: get_spatial_extent (player: spatial_temporal_specialist)
2026-07-15 13:46:58,243 - INFO - root -   Step 4: get_temporal_extent (player: spatial_temporal_specialist)
2026-07-15 13:46:58,243 - INFO - root -   Step 5: generate_metadata (player: croissant_inaturalist_metadata_generator)


In [4]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

✅ Plan dataflow is valid.


In [5]:
plan_dicts

[{'task': 'detect_spatial_columns',
  'player': 'spatial_temporal_specialist',
  'rationale': 'Detect spatial columns in the CSV to determine the bounding box for spatialCoverage.',
  'target_resources': [],
  'inputs': {},
  'outputs': ['spatial_detection']},
 {'task': 'detect_temporal_columns',
  'player': 'spatial_temporal_specialist',
  'rationale': 'Detect temporal columns in the CSV to determine the date range for temporalCoverage.',
  'target_resources': [],
  'inputs': {},
  'outputs': ['temporal_detection']},
 {'task': 'get_spatial_extent',
  'player': 'spatial_temporal_specialist',
  'rationale': 'Calculate the spatial bounding box from detected coordinate columns.',
  'target_resources': [],
  'inputs': {'spatial_detection': 'spatial_detection'},
  'outputs': ['spatial_extent']},
 {'task': 'get_temporal_extent',
  'player': 'spatial_temporal_specialist',
  'rationale': 'Calculate the temporal extent from detected date/time columns.',
  'target_resources': [],
  'inputs': {'t

In [6]:
plan.pretty_print()

Plan Steps:
Step 0: detect_spatial_columns
  Rationale: Detect spatial columns in the CSV to determine the bounding box for spatialCoverage.
  Required Artifacts: {}
  Produced Artifacts: ['spatial_detection']
Step 1: detect_temporal_columns
  Rationale: Detect temporal columns in the CSV to determine the date range for temporalCoverage.
  Required Artifacts: {}
  Produced Artifacts: ['temporal_detection']
Step 2: get_spatial_extent
  Rationale: Calculate the spatial bounding box from detected coordinate columns.
  Required Artifacts: {'spatial_detection': 'spatial_detection'}
  Produced Artifacts: ['spatial_extent']
Step 3: get_temporal_extent
  Rationale: Calculate the temporal extent from detected date/time columns.
  Required Artifacts: {'temporal_detection': 'temporal_detection'}
  Produced Artifacts: ['temporal_extent']
Step 4: generate_metadata
  Rationale: Generate the final iNaturalist Croissant metadata using the spatial and temporal extent artifacts.
  Required Artifacts: {'

In [7]:
from src.orchestrator.plan_executor import PlanExecutor
from src.tools.context_tools import register_context

context_key = "ctx_biota_multi"
register_context(context_key, data_context)
executor = PlanExecutor(topology_name="single")
result = executor.execute(
    plan=plan,
    context=data_context,
    context_key=context_key,
    metadata_standard=metadata_standard,
    metadata_standard_name=STANDARD_NAME
)

2026-07-15 13:47:08,099 - INFO - root - PlanExecutor initialized with topology: single
2026-07-15 13:47:08,099 - INFO - root -   Players per step: 1
2026-07-15 13:47:08,099 - INFO - root -   Debate rounds: 0
2026-07-15 13:47:08,099 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-07-15 13:47:08,100 - INFO - root - Using structured output schema: CroissantINaturalistMetadata
2026-07-15 13:47:08,100 - INFO - root - ============================================================
2026-07-15 13:47:08,100 - INFO - root - STARTING PLAN EXECUTION
2026-07-15 13:47:08,100 - INFO - root - Context: test
2026-07-15 13:47:08,100 - INFO - root - Type: single_csv
2026-07-15 13:47:08,101 - INFO - root - Resources: ['file_1']
2026-07-15 13:47:08,101 - INFO - root - Steps: 5
2026-07-15 13:47:08,101 - INFO - root - ============================================================


/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-15 13:47:14,145 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:47:14,147 - INFO - root - Player 'spatial_temporal_specialist_1': blocking finish — still need: detect_temporal_columns, detect_spatial_columns
2026-07-15 13:47:21,672 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:47:21,673 - INFO - root - Player 'spatial_temporal_specialist_1': blocking finish — still need: detect_temporal_columns, detect_spatial_columns
2026-07-15 13:47:23,928 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:47:23,937 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 3 — model requested 1 tool(s): detect_temporal_columns
2026-07-15 13:47:25,961 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:47:25,969 -

/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-15 13:48:00,195 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:48:00,197 - INFO - root - Player 'spatial_temporal_specialist_1': blocking finish — still need: detect_temporal_columns, detect_spatial_columns
2026-07-15 13:48:01,198 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:48:01,204 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 2 — model requested 1 tool(s): detect_temporal_columns
2026-07-15 13:48:01,858 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:48:01,865 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 3 — model requested 1 tool(s): detect_spatial_columns
2026-07-15 13:48:04,898 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:48:04,900 -

...

2026-07-15 13:48:19,437 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:48:19,438 - INFO - root - Player 'spatial_temporal_specialist_1': blocking finish — still need: detect_temporal_columns, detect_spatial_columns
2026-07-15 13:48:20,839 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:48:20,844 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 2 — model requested 1 tool(s): detect_temporal_columns
2026-07-15 13:48:21,588 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:48:21,596 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 3 — model requested 1 tool(s): detect_spatial_columns
2026-07-15 13:48:23,812 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:48:23,813 -

/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-15 13:49:19,828 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:49:19,829 - INFO - root - Player 'spatial_temporal_specialist_1': blocking finish — still need: detect_temporal_columns, detect_spatial_columns
2026-07-15 13:49:22,830 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:49:22,836 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 2 — model requested 1 tool(s): detect_temporal_columns
2026-07-15 13:49:27,244 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:49:27,251 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 3 — model requested 1 tool(s): detect_spatial_columns
2026-07-15 13:49:37,450 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:49:37,451 -

/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-15 13:50:06,075 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:50:06,077 - INFO - root -   Player 'croissant_inaturalist_metadata_generator_1' completed execution
2026-07-15 13:50:06,077 - INFO - root -     Output: {   "spatialCoverage": {     "min_lat": 18.6,     "min_lon": 1.0,     "max_lat": 31.2,     "max_lon": 101.0   },   "temporalCoverage": {     "from": "2014-01-01T00:00:00+00:00",     "to": "2014-01-01T...
2026-07-15 13:50:06,077 - INFO - root - Single player, skipping debate
2026-07-15 13:50:06,078 - INFO - root - --- STEP 4: SYNTHESIS ---
2026-07-15 13:50:06,078 - INFO - root -   Using structured output with schema: CroissantINaturalistMetadata
2026-07-15 13:50:10,430 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-15 13:50:10,436 - INFO - root -   Structured output validated successfully
2026-07-15 13:50:10,436 - INFO - root -   Synthesis 

In [ ]:
from pprint import pprint
pprint(result.final_workspace['metadata_output'])

In [8]:
executor.list_tools_called()

['detect_temporal_columns',
 'detect_spatial_columns',
 'get_temporal_extent',
 'get_spatial_extent',
 'detect_temporal_columns',
 'detect_spatial_columns',
 'get_temporal_extent',
 'get_spatial_extent',
 'detect_temporal_columns',
 'detect_spatial_columns',
 'get_temporal_extent',
 'get_spatial_extent',
 'detect_temporal_columns',
 'detect_spatial_columns',
 'get_temporal_extent',
 'get_spatial_extent']

In [11]:
executor.find_tool_calls('detect_spatial_columns')

[{'agent': 'spatial_temporal_specialist_1',
  'input': {'context_key': 'ctx_biota_multi', 'resource': 'file_1'},
  'output': {'resource': 'file_1',
   'spatial_column_count': 15,
   'spatial_columns': {'Sample ID': {'name_suggests_spatial': False,
     'detected_type': 'possible_longitude',
     'sample_values': ['1', '2', '3', '4', '5']},
    'Temp': {'name_suggests_spatial': False,
     'detected_type': 'possible_latitude',
     'sample_values': ['21.6', '21.6', '20.7', '20.7', '20.7']},
    'Sal': {'name_suggests_spatial': False,
     'detected_type': 'possible_latitude',
     'sample_values': ['39', '39', '39', '39', '39']},
    'E. fimbriata oocyte dm': {'name_suggests_spatial': False,
     'detected_type': 'possible_latitude',
     'sample_values': ['38.4', '35.3', '27.0', '33.4', '29.0']},
    'E. fimbriata oocyte vol': {'name_suggests_spatial': False,
     'detected_type': 'possible_latitude',
     'sample_values': ['0.4', '0.3', '0.4', '0.3', '0.5']},
    'Lipids': {'name_sugg

In [10]:
executor.find_tool_calls('get_spatial_extent')

[{'agent': 'spatial_temporal_specialist_1',
  'input': {'context_key': 'ctx_biota_multi',
   'resource': 'file_1',
   'lat_column': 'Temp',
   'lon_column': 'Sample ID'},
  'output': {'resource': 'file_1',
   'lat_column': 'Temp',
   'lon_column': 'Sample ID',
   'valid_point_count': 101,
   'bounding_box': {'min_lat': 18.6,
    'max_lat': 31.2,
    'min_lon': 1.0,
    'max_lon': 101.0},
   'center': {'lat': 25.655247524752465, 'lon': 51.0}}},
 {'agent': 'spatial_temporal_specialist_1',
  'input': {'context_key': 'ctx_biota_multi',
   'resource': 'file_1',
   'lat_column': 'Temp',
   'lon_column': 'Sample ID'},
  'output': {'resource': 'file_1',
   'lat_column': 'Temp',
   'lon_column': 'Sample ID',
   'valid_point_count': 101,
   'bounding_box': {'min_lat': 18.6,
    'max_lat': 31.2,
    'min_lon': 1.0,
    'max_lon': 101.0},
   'center': {'lat': 25.655247524752465, 'lon': 51.0}}},
 {'agent': 'spatial_temporal_specialist_1',
  'input': {'context_key': 'ctx_biota_multi',
   'resource':

## Validate iNaturalist result against CSV ground truth

All files under `data/experiment/inaturalist_100_species_1000_obs/...` share the same 50 GBIF/iNaturalist occurrence columns. Ground truth is computed from the full CSV using:

- `decimalLatitude` / `decimalLongitude` → `spatialCoverage` bounding box
- `eventDate` → `temporalCoverage` (`from` / `to`)

In [ ]:
import json
from pathlib import Path

import pandas as pd

SPATIAL_KEYS = ('min_lat', 'min_lon', 'max_lat', 'max_lon')
COORD_TOLERANCE = 1e-5  # ~1 m


def inat_ground_truth(csv_path: str | Path) -> dict:
    """Ground truth from a full iNaturalist occurrence CSV."""
    df = pd.read_csv(csv_path)
    lat = pd.to_numeric(df['decimalLatitude'], errors='coerce')
    lon = pd.to_numeric(df['decimalLongitude'], errors='coerce')
    valid = lat.notna() & lon.notna()

    dates = pd.to_datetime(df['eventDate'], errors='coerce', utc=True)
    valid_dates = dates.dropna()

    return {
        'spatialCoverage': {
            'min_lat': float(lat[valid].min()),
            'min_lon': float(lon[valid].min()),
            'max_lat': float(lat[valid].max()),
            'max_lon': float(lon[valid].max()),
        },
        'temporalCoverage': {
            'from': valid_dates.min().isoformat(),
            'to': valid_dates.max().isoformat(),
        },
        'stats': {
            'rows': len(df),
            'valid_coords': int(valid.sum()),
            'valid_dates': int(valid_dates.shape[0]),
        },
    }


def extract_predicted_metadata(result) -> dict | None:
    if result is None:
        return None
    workspace = result.final_workspace or {}
    for candidate in (result.final_metadata, workspace.get('metadata_output')):
        if isinstance(candidate, dict) and {'spatialCoverage', 'temporalCoverage'}.issubset(candidate.keys()):
            return candidate
    return None


def compare_spatial(gt: dict, pred: dict, tol: float = COORD_TOLERANCE) -> list[str]:
    issues = []
    for key in SPATIAL_KEYS:
        gt_val, pred_val = gt[key], pred[key]
        if abs(gt_val - pred_val) > tol:
            issues.append(f'{key}: gt={gt_val}, pred={pred_val}, diff={pred_val - gt_val}')
    return issues


def compare_temporal(gt: dict, pred: dict) -> list[str]:
    gt_from = pd.to_datetime(gt['from'], utc=True)
    gt_to = pd.to_datetime(gt['to'], utc=True)
    pred_from = pd.to_datetime(pred['from'], utc=True)
    pred_to = pd.to_datetime(pred['to'], utc=True)

    issues = []
    if gt_from != pred_from:
        issues.append(f"from: gt={gt_from.isoformat()}, pred={pred_from.isoformat()}")
    if gt_to != pred_to:
        issues.append(f"to: gt={gt_to.isoformat()}, pred={pred_to.isoformat()}")
    return issues


csv_path = Path(next(iter(inat_source.values())))
gt = inat_ground_truth(csv_path)
pred = extract_predicted_metadata(result)

print(f'Source CSV: {csv_path.name}')
print(f"Rows: {gt['stats']['rows']} | valid coords: {gt['stats']['valid_coords']} | valid eventDate: {gt['stats']['valid_dates']}")
print()
print('Ground truth:')
print(json.dumps({k: gt[k] for k in ('spatialCoverage', 'temporalCoverage')}, indent=2))
print()
print('Predicted:')
print(json.dumps(pred, indent=2, default=str) if pred else None)
print()

if pred is None:
    print('Overall: FAIL (no schema-shaped metadata in result)')
else:
    spatial_issues = compare_spatial(gt['spatialCoverage'], pred['spatialCoverage'])
    temporal_issues = compare_temporal(gt['temporalCoverage'], pred['temporalCoverage'])

    print('Spatial match:', 'YES' if not spatial_issues else 'NO')
    for issue in spatial_issues:
        print(f'  - {issue}')

    print('Temporal match:', 'YES' if not temporal_issues else 'NO')
    for issue in temporal_issues:
        print(f'  - {issue}')

    print()
    print('Overall:', 'PASS' if not spatial_issues and not temporal_issues else 'FAIL')